#  Notebook 3 – Missing Value Handling


# 1. What are Missing Values?


## Step 1 – Understand Concept

Missing values are data points where information is not available or has not been recorded.

In Pandas, missing values are commonly represented as:

- NaN
- None
- NaT for missing date/time values

Missing values can occur in numerical, categorical, and date/time columns.

Before handling missing values, we should understand why they are missing and how much data is affected.


## Step 2 – Demonstrate Concept

Consider a customer dataset:

| Customer | Age | MonthlyCharges |
|----------|-----|----------------|
| C001 | 25 | 50.5 |
| C002 | NaN | 60.2 |
| C003 | 35 | NaN |

Here:

- Age is missing for C002.
- MonthlyCharges is missing for C003.

We should not immediately replace these values.

First, we should investigate the reason and percentage of missing data.


## Step 3 – Implement the Concept

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Customer": ["C001", "C002", "C003", "C004"],
    "Age": [25, np.nan, 35, 40],
    "MonthlyCharges": [50.5, 60.2, np.nan, 75.8]
})

print(df)
print("\nMissing Values:")
print(df.isna().sum())


  Customer   Age  MonthlyCharges
0     C001  25.0            50.5
1     C002   NaN            60.2
2     C003  35.0             NaN
3     C004  40.0            75.8

Missing Values:
Customer          0
Age               1
MonthlyCharges    1
dtype: int64


## Result / Observation

The `isna().sum()` output shows how many missing values are present in each column.

Missing-value handling should begin with detection and analysis rather than immediate filling.

# 2. Why Missing Values Occur


## Step 1 – Understand Concept

Missing values can occur for many reasons.

Common causes include:

- User did not provide information
- Data collection failure
- Sensor failure
- System error
- Information was not applicable
- Data was lost during data integration
- Database fields were optional
- Privacy-related non-response

The reason for missingness is important because it influences the appropriate treatment.


## Step 2 – Demonstrate Concept

Example:

A customer may not provide their income.

This could happen because:

- The customer chose not to provide it.
- The field was not mandatory.
- The system failed to capture it.

These situations have different meanings.

Therefore, blindly replacing every missing value with the mean may introduce incorrect information.


## Step 3 – Implement the Concept

In [2]:
df = pd.DataFrame({
    "Customer": ["C001", "C002", "C003", "C004"],
    "Income": [30000, np.nan, 50000, np.nan],
    "Age": [25, 30, np.nan, 40]
})

print(df)
print("\nMissing Value Count:")
print(df.isna().sum())

  Customer   Income   Age
0     C001  30000.0  25.0
1     C002      NaN  30.0
2     C003  50000.0   NaN
3     C004      NaN  40.0

Missing Value Count:
Customer    0
Income      2
Age         1
dtype: int64


## Result / Observation

Missing values exist in both `Income` and `Age`.

Before choosing an imputation method, we should understand the reason and pattern of the missing values.

# 3. MCAR – Missing Completely At Random


## Step 1 – Understand Concept

MCAR means Missing Completely At Random.

The missingness is not related to:

- The missing value itself
- Other variables in the dataset

The missing values occur randomly.

MCAR is generally the simplest missing-data situation to handle.


## Step 2 – Demonstrate Concept

Suppose some customer records lost their phone number because of a temporary system failure.

The failure happened randomly and was not related to:

- Customer age
- Salary
- Contract
- Churn status

This is an example of MCAR.

In such cases, dropping a small number of affected records may sometimes be acceptable.


## Step 3 – Implement the Concept

In [3]:
df = pd.DataFrame({
    "Age": [25, 30, np.nan, 40, 35],
    "Income": [30000, 40000, 50000, 60000, 35000]
})

print(df)
print("\nMissing values:")
print(df.isna().sum())

    Age  Income
0  25.0   30000
1  30.0   40000
2   NaN   50000
3  40.0   60000
4  35.0   35000

Missing values:
Age       1
Income    0
dtype: int64


## Result / Observation

The dataset contains a missing Age value.

If the missingness is genuinely random and the percentage is very small, removing the affected record may be reasonable.

However, we should verify the missingness pattern before making this decision.

# 4. MAR – Missing At Random


## Step 1 – Understand Concept

MAR means Missing At Random.

The probability of a value being missing depends on other observed variables, but not directly on the missing value itself after considering those variables.

For example, income may be missing more frequently for customers in a particular age group.


## Step 2 – Demonstrate Concept

Suppose younger customers are less likely to provide their income.

Here:

- Age is observed.
- Income is missing for some customers.
- Missingness depends on Age.

This can be considered an example of MAR.

Group-based or model-based imputation may be more appropriate than simply using the overall mean.


## Step 3 – Implement the Concept

In [4]:
df = pd.DataFrame({
    "Age": [22, 24, 25, 45, 50, 52],
    "Income": [25000, np.nan, 28000, 60000, np.nan, 65000]
})

print(df)

print("\nMissing Income by Age Group:")

df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 30, 60],
    labels=["Young", "Older"]
)

print(df.groupby("AgeGroup", observed=True)["Income"].apply(lambda x: x.isna().sum()))

   Age   Income
0   22  25000.0
1   24      NaN
2   25  28000.0
3   45  60000.0
4   50      NaN
5   52  65000.0

Missing Income by Age Group:
AgeGroup
Young    1
Older    1
Name: Income, dtype: int64


## Result / Observation

The missingness can be investigated across observed groups such as age groups.

When missingness depends on observed variables, group-based or model-based approaches may preserve the underlying structure better than overall mean imputation.

# 5. MNAR – Missing Not At Random


## Step 1 – Understand Concept

MNAR means Missing Not At Random.

The probability of a value being missing depends on the missing value itself or an unobserved factor related to it.

For example, people with very high income may be less likely to report their income.

Here, the missingness may be related to the actual income value.



## Step 2 – Demonstrate Concept

Suppose customers with very high salaries are less willing to disclose their income.

The missingness is related to the income itself.

Using the overall mean could underestimate the actual income distribution.

MNAR is more difficult to handle and may require domain knowledge and specialized statistical approaches.



## Step 3 – Implement the Concept

In [5]:
df = pd.DataFrame({
    "Customer": ["C001", "C002", "C003", "C004", "C005"],
    "Income": [30000, 35000, np.nan, 45000, np.nan]
})

print(df)

print("\nMissing Income:")
print(df["Income"].isna().sum())

  Customer   Income
0     C001  30000.0
1     C002  35000.0
2     C003      NaN
3     C004  45000.0
4     C005      NaN

Missing Income:
2


## Result / Observation

The code detects missing values, but code alone cannot prove that the missingness is MNAR.

Determining MNAR usually requires domain knowledge or additional information about why values are missing.

# 6. Detecting Missing Values


## Step 1 – Understand Concept

Before handling missing values, we must identify:

- Which columns contain missing values
- Number of missing values
- Percentage of missing values
- Rows containing missing values

Pandas provides methods such as:

- `isna()`
- `isnull()`
- `notna()`
- `notnull()`



## Step 2 – Demonstrate Concept

For a dataset, we may want to answer:

1. Does the column contain missing values?
2. How many values are missing?
3. Which rows contain missing values?

This gives us an initial understanding of the data quality problem.


## Step 3 – Implement the Concept




In [6]:
df = pd.DataFrame({
    "Age": [25, np.nan, 35, 40],
    "Salary": [30000, 40000, np.nan, 60000],
    "City": ["Chennai", "Mumbai", np.nan, "Delhi"]
})

print("Missing value count:")
print(df.isna().sum())

print("\nRows containing missing values:")
print(df[df.isna().any(axis=1)])

Missing value count:
Age       1
Salary    1
City      1
dtype: int64

Rows containing missing values:
    Age   Salary    City
1   NaN  40000.0  Mumbai
2  35.0      NaN     NaN


## Result / Observation

The missing values and affected rows have been identified.

This information helps us decide which missing-value technique is appropriate.

# 7. Missing Value Percentage


## Step 1 – Understand Concept

Missing-value percentage tells us what proportion of a column is missing.

Formula:

Missing Percentage =
(Number of Missing Values / Total Number of Rows) × 100

Percentage is more useful than only the count because datasets can have different sizes.


## Step 2 – Demonstrate Concept

Suppose a dataset contains 1,000 rows.

If 20 values are missing:

20 / 1000 × 100 = 2%

If 600 values are missing:

600 / 1000 × 100 = 60%

A 2% missing rate and a 60% missing rate require very different decisions.


## Step 3 – Implement the Concept

In [7]:
missing_percentage = df.isna().mean() * 100

print("Missing Value Percentage:")
print(missing_percentage)

Missing Value Percentage:
Age       25.0
Salary    25.0
City      25.0
dtype: float64


## Result / Observation

The output shows the percentage of missing values in every column.

This helps us decide whether to:

- Drop rows
- Drop a column
- Impute values
- Investigate the source of missingness

# 8. Dropping Rows


## Step 1 – Understand Concept

Dropping rows means removing records that contain missing values.

Pandas provides:

`dropna()`

Row deletion can be useful when:

- Only a small percentage of rows are affected.
- The missingness is random.
- The affected records are not important.
- Enough data remains after deletion.


## Step 2 – Demonstrate Concept

Suppose a dataset has 10,000 records and only 20 records contain missing values.

Removing those 20 rows may have little impact.

However, if 4,000 records contain missing values, dropping all of them could remove too much useful information.


## Step 3 – Implement the Concept

In [8]:
df = pd.DataFrame({
    "Age": [25, np.nan, 35, 40],
    "Salary": [30000, 40000, np.nan, 60000]
})

print("Before:")
print(df)

df_dropped = df.dropna()

print("\nAfter dropping rows:")
print(df_dropped)

Before:
    Age   Salary
0  25.0  30000.0
1   NaN  40000.0
2  35.0      NaN
3  40.0  60000.0

After dropping rows:
    Age   Salary
0  25.0  30000.0
3  40.0  60000.0


## Result / Observation

Rows containing missing values were removed.

### When to use:
- Small percentage of missing rows
- Missingness is random
- Sufficient data remains

### When not to use:
- Large amount of missing data
- Small dataset
- Important records contain missing values

### Advantage:
Simple and easy.

### Limitation:
Can cause information loss and reduce dataset size.

# 9. Dropping Columns


## Step 1 – Understand Concept

A column may be removed when it contains a very high percentage of missing values and does not provide enough useful information.

However, we should not remove a column only because it contains missing values.

The importance of the feature and business meaning must also be considered.


## Step 2 – Demonstrate Concept

Suppose:

CustomerID → 0% missing
Age → 5% missing
Income → 15% missing
OptionalSurveyField → 95% missing

The OptionalSurveyField may be a candidate for removal if it provides little useful information.

However, a feature with 95% missing values could still be important in some business situations, so the decision should be documented.


## Step 3 – Implement the Concept

In [9]:
df = pd.DataFrame({
    "Age": [25, 30, np.nan, 40],
    "Income": [30000, np.nan, 50000, 60000],
    "OptionalField": [np.nan, np.nan, np.nan, 100]
})

print("Missing percentages:")
print(df.isna().mean() * 100)

df_dropped = df.drop(columns=["OptionalField"])

print("\nAfter dropping column:")
print(df_dropped)

Missing percentages:
Age              25.0
Income           25.0
OptionalField    75.0
dtype: float64

After dropping column:
    Age   Income
0  25.0  30000.0
1  30.0      NaN
2   NaN  50000.0
3  40.0  60000.0


## Result / Observation

The column with a very high missing percentage was removed as an example.

### When to use:
- Very high missing percentage
- Low business importance
- Feature does not provide enough useful information

### When not to use:
- Important business feature
- Missing values can be reliably imputed

### Advantage:
Simplifies the dataset.

### Limitation:
Potentially removes useful information.

# 10. Mean Imputation


## Step 1 – Understand Concept

Mean imputation replaces missing numerical values with the mean of the available values.

Example:

10, 20, NaN, 30

Mean = 20

After imputation:

10, 20, 20, 30

Mean imputation is mainly suitable for numerical data that is reasonably symmetric and does not contain strong outliers.


## Step 2 – Demonstrate Concept

Suppose MonthlyCharges has a roughly symmetric distribution and only a small number of values are missing.

The mean can be used as a simple replacement.

However, if the data is highly skewed or contains extreme values, the mean can be strongly affected.

In such cases, median may be better.


## Step 3 – Implement the Concept

In [10]:
df = pd.DataFrame({
    "MonthlyCharges": [50, 60, np.nan, 70, 80]
})

mean_value = df["MonthlyCharges"].mean()

df["MonthlyCharges"] = df["MonthlyCharges"].fillna(mean_value)

print(df)

   MonthlyCharges
0            50.0
1            60.0
2            65.0
3            70.0
4            80.0


## Result / Observation

The missing value was replaced with the column mean.

### When to use:
- Numerical data
- Approximately symmetric distribution
- Few missing values
- No strong outliers

### When not to use:
- Highly skewed data
- Strong outliers
- Missingness has an important pattern

### Advantage:
Simple and fast.

### Limitation:
Can reduce variance and distort the distribution.

Mean should NOT be used automatically for every missing numerical value.

# 11. Median Imputation

## Step 1 – Understand Concept

Median imputation replaces missing numerical values with the median of the available values.

Median is the middle value after sorting the data.

It is less affected by extreme values than the mean.



## Step 2 – Demonstrate Concept

Consider:

20, 25, 30, 35, 500

The value 500 is an extreme value.

The mean is strongly affected by 500.

The median remains closer to the typical values.

Therefore, median imputation is often preferred for skewed numerical features or features containing outliers.


## Step 3 – Implement the Concept

In [11]:
df = pd.DataFrame({
    "Income": [20000, 25000, np.nan, 30000, 100000]
})

median_value = df["Income"].median()

df["Income"] = df["Income"].fillna(median_value)

print(df)

     Income
0   20000.0
1   25000.0
2   27500.0
3   30000.0
4  100000.0


## Result / Observation

The missing income was replaced using the median.

### When to use:
- Numerical data
- Skewed distributions
- Data containing outliers

### When not to use:
- When the feature distribution or business meaning makes median inappropriate

### Advantage:
Robust to outliers.

### Limitation:
Can reduce natural variation and may not capture relationships between variables.

# 12. Mode Imputation


## Step 1 – Understand Concept

Mode is the most frequently occurring value.

Mode imputation replaces missing categorical values with the most common category.

It is mainly used for categorical variables.


## Step 2 – Demonstrate Concept

Suppose Contract contains:

Month-to-month
One year
Month-to-month
NaN
Month-to-month

The mode is:

Month-to-month

Therefore, the missing value can be replaced with Month-to-month.

However, if the categories are highly imbalanced, mode imputation may make the imbalance worse.


## Step 3 – Implement the Concept

In [12]:
df = pd.DataFrame({
    "Contract": [
        "Month-to-month",
        "One year",
        "Month-to-month",
        np.nan,
        "Month-to-month"
    ]
})

mode_value = df["Contract"].mode()[0]

df["Contract"] = df["Contract"].fillna(mode_value)

print(df)

         Contract
0  Month-to-month
1        One year
2  Month-to-month
3  Month-to-month
4  Month-to-month


## Result / Observation

The missing categorical value was replaced with the most frequent category.

### When to use:
- Categorical variables
- Small amount of missing data
- Mode represents a reasonable default

### When not to use:
- High missing percentage
- Highly imbalanced categories
- Missingness contains meaningful information

### Advantage:
Simple and easy to implement.

### Limitation:
Can increase the frequency of the dominant category and distort category distribution.

# 13. Constant Value Imputation


## Step 1 – Understand Concept

Constant imputation replaces missing values with a predefined value.

Examples:

Numerical:
0

Categorical:
"Unknown"

"Not Provided"

The constant should have a meaningful reason.



## Step 2 – Demonstrate Concept

Suppose a customer did not provide their preferred contact method.

Instead of guessing the category, we can use:

"Unknown"

This preserves the fact that the information was not available.

For some numerical features, 0 may be meaningful, but we should not use 0 unless it makes business sense.



## Step 3 – Implement the Concept

In [13]:
df = pd.DataFrame({
    "ContactMethod": ["Email", np.nan, "Phone", "Email"]
})

df["ContactMethod"] = df["ContactMethod"].fillna("Unknown")

print(df)

  ContactMethod
0         Email
1       Unknown
2         Phone
3         Email


## Result / Observation

Missing values were replaced with `"Unknown"`.

### When to use:
- Missingness itself carries meaning
- Categorical variables
- Business-defined default values

### When not to use:
- Arbitrary numerical replacement
- When the constant creates unrealistic values

### Advantage:
Simple and preserves an explicit missing category.

### Limitation:
Can introduce an artificial category or distort numerical distributions.

# 14. Forward Fill

## Step 1 – Understand Concept

Forward fill (`ffill`) fills a missing value using the previous available value.

Example:

100
200
NaN
400

After forward fill:

100
200
200
400

It is mainly useful for ordered or time-series data where the previous value is reasonably expected to continue.


## Step 2 – Demonstrate Concept

Suppose a sensor records temperature every hour.

If one reading is missing, using the previous reading may be reasonable when the value changes slowly.

It is not appropriate for every dataset.


## Step 3 – Implement the Concept

In [14]:
df = pd.DataFrame({
    "Temperature": [30.2, 30.5, np.nan, np.nan, 31.0]
})

df["Temperature"] = df["Temperature"].ffill()

print(df)

   Temperature
0         30.2
1         30.5
2         30.5
3         30.5
4         31.0


## Result / Observation

The missing values were filled using the most recent previous value.

### When to use:
- Time-series data
- Ordered observations
- Values are expected to remain stable for short periods

### When not to use:
- Unordered customer data
- Values change rapidly
- Long consecutive missing periods

### Advantage:
Simple and preserves temporal continuity.

### Limitation:
Can propagate an incorrect or outdated value.

# 15. Backward Fill


## Step 1 – Understand Concept

Backward fill (`bfill`) fills a missing value using the next available value.

Example:

100
NaN
200

After backward fill:

100
200
200

It can be useful for ordered or time-series data when the next observation is a suitable replacement.


## Step 2 – Demonstrate Concept

Suppose a sensor reading is missing but the next reading is known.

If the values are expected to remain relatively stable, the next reading can be used.

The technique should not be blindly applied to unordered data.



## Step 3 – Implement the Concept

In [15]:
df = pd.DataFrame({
    "Temperature": [30.2, np.nan, np.nan, 31.0, 31.5]
})

df["Temperature"] = df["Temperature"].bfill()

print(df)

   Temperature
0         30.2
1         31.0
2         31.0
3         31.0
4         31.5


## Result / Observation

Missing values were filled using the next available value.

### When to use:
- Ordered/time-series data
- Next value is a reasonable representation

### When not to use:
- Unordered data
- Long missing sequences
- When future values should not influence past observations

### Advantage:
Maintains continuity.

### Limitation:
Can introduce future information into earlier records, which can cause data leakage in some predictive/time-series scenarios.

# 16. Interpolation


## Step 1 – Understand Concept

Interpolation estimates missing numerical values using surrounding known values.

For example:

10
NaN
30

Linear interpolation estimates the missing value as:

20

Interpolation is mainly useful for ordered numerical or time-series data.


## Step 2 – Demonstrate Concept

Suppose temperature readings are:

10:00 → 30
11:00 → Missing
12:00 → 34

A linear interpolation can estimate the 11:00 value based on the surrounding observations.

This can be more realistic than simply using the overall mean.



## Step 3 – Implement the Concept

In [16]:
df = pd.DataFrame({
    "Temperature": [30.0, np.nan, 34.0, 35.0]
})

df["Temperature"] = df["Temperature"].interpolate()

print(df)

   Temperature
0         30.0
1         32.0
2         34.0
3         35.0


## Result / Observation

The missing value was estimated using surrounding numerical values.

### When to use:
- Ordered numerical data
- Time-series data
- Values change gradually

### When not to use:
- Unordered categorical data
- Highly irregular data
- When interpolation assumptions are not valid

### Advantage:
Uses local information and can preserve trends.

### Limitation:
The estimated value is based on assumptions about how the data changes.

# 17. Group-Based Imputation


## Step 1 – Understand Concept

Group-based imputation replaces missing values using statistics calculated within a meaningful group.

For example:

- Average salary by department
- Median charges by contract type
- Median income by city

This can preserve group-level differences better than using one overall value.



## Step 2 – Demonstrate Concept

Suppose customers belong to different contract types.

MonthlyCharges may be different for each contract group.

Using one overall median could ignore these differences.

Instead, we can calculate the median MonthlyCharges separately for each Contract type.



## Step 3 – Implement the Concept

In [17]:
df = pd.DataFrame({
    "Contract": [
        "Month-to-month",
        "Month-to-month",
        "One year",
        "One year",
        "Two year"
    ],
    "MonthlyCharges": [
        50,
        np.nan,
        70,
        np.nan,
        80
    ]
})

df["MonthlyCharges"] = df.groupby(
    "Contract"
)["MonthlyCharges"].transform(
    lambda x: x.fillna(x.median())
)

print(df)

         Contract  MonthlyCharges
0  Month-to-month            50.0
1  Month-to-month            50.0
2        One year            70.0
3        One year            70.0
4        Two year            80.0


## Result / Observation

Missing values were handled using the median within each contract group.

### When to use:
- Strong group-level differences exist
- A meaningful grouping variable is available
- Group contains enough valid observations

### When not to use:
- Groups are extremely small
- No meaningful grouping variable exists

### Advantage:
Preserves group-level patterns better than global imputation.

### Limitation:
Small groups may have unreliable statistics.

# 18. KNN Imputation


## Step 1 – Understand Concept

KNN means K-Nearest Neighbors.

KNN imputation estimates a missing value using similar records.

Instead of using one overall mean, it looks for observations that are close to the incomplete record based on other features.

The missing value is then estimated from those similar observations.


## Step 2 – Demonstrate Concept

Suppose a customer's MonthlyCharges value is missing.

The algorithm can look for customers with similar:

- Tenure
- TotalCharges
- Other numerical characteristics

If those similar customers have MonthlyCharges values, they can help estimate the missing value.

This can preserve relationships between features better than simple mean imputation.



## Step 3 – Implement the Concept

In [18]:
from sklearn.impute import KNNImputer

df = pd.DataFrame({
    "Age": [25, 30, 35, 40, 45],
    "Income": [30000, 40000, np.nan, 60000, 70000],
    "MonthlyCharges": [50, 60, 70, np.nan, 90]
})

imputer = KNNImputer(n_neighbors=2)

df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

print(df_imputed)

    Age   Income  MonthlyCharges
0  25.0  30000.0            50.0
1  30.0  40000.0            60.0
2  35.0  50000.0            70.0
3  40.0  60000.0            80.0
4  45.0  70000.0            90.0


## Result / Observation

KNN imputation estimated missing numerical values using nearby observations.

### When to use:
- Numerical data
- Features have meaningful relationships
- Dataset is not extremely large

### When not to use:
- Very large datasets where computation becomes expensive
- Features are not meaningfully comparable
- Poorly scaled numerical features

### Advantage:
Can preserve relationships between features.

### Limitation:
Computationally more expensive and sensitive to feature scaling and choice of neighbors.

# 19. Iterative Imputation

## Step 1 – Understand Concept

Iterative imputation estimates missing values by modeling each incomplete feature using other available features.

The process is repeated multiple times so that the estimated values can improve based on relationships between variables.

It is more sophisticated than simple mean or median imputation.


 ## Step 2 – Demonstrate Concept

Suppose we have:

- Age
- Income
- MonthlyCharges

Income has missing values.

Iterative imputation can use relationships between Age and MonthlyCharges to estimate Income.

The process is repeated until the estimates become stable or the maximum number of iterations is reached.


## Step 3 – Implement the Concept

In [19]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df = pd.DataFrame({
    "Age": [25, 30, 35, 40, 45],
    "Income": [30000, np.nan, 50000, 60000, np.nan],
    "MonthlyCharges": [50, 60, np.nan, 80, 90]
})

imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)

df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

print(df_imputed)

    Age        Income  MonthlyCharges
0  25.0  30000.000000            50.0
1  30.0  39999.999999            60.0
2  35.0  50000.000000            70.0
3  40.0  60000.000000            80.0
4  45.0  69999.999999            90.0


## Result / Observation

Iterative imputation estimated missing values using relationships between multiple numerical features.

### When to use:
- Multiple numerical features are related
- Missingness is moderate
- A more sophisticated method is justified

### When not to use:
- Very small datasets
- Features have weak relationships
- Simple imputation is already sufficient

### Advantage:
Can capture relationships between variables.

### Limitation:
More complex, computationally expensive, and model assumptions can affect the results.

# 20. Comparing Missing Value Techniques


## Step 1 – Understand Concept

There is no single best missing-value technique.

The correct method depends on:

- Data type
- Missing percentage
- Distribution
- Outliers
- Missingness mechanism
- Relationship between features
- Dataset size
- Business meaning


## Step 2 – Demonstrate Concept

| Technique | Suitable Situation |
|---|---|
| Drop Rows | Very few missing records |
| Drop Columns | Very high missing percentage and low importance |
| Mean | Symmetric numerical data |
| Median | Skewed numerical data / outliers |
| Mode | Categorical data |
| Constant | Meaningful default category/value |
| Forward Fill | Ordered/time-series data |
| Backward Fill | Ordered data where next value is appropriate |
| Interpolation | Continuous ordered numerical data |
| Group-Based | Strong group differences |
| KNN | Related numerical features |
| Iterative | Strong relationships among multiple features |


## Step 3 – Implement the Concept

In [20]:
comparison = pd.DataFrame({
    "Technique": [
        "Drop Rows",
        "Drop Columns",
        "Mean",
        "Median",
        "Mode",
        "Constant",
        "Forward Fill",
        "Backward Fill",
        "Interpolation",
        "Group-Based",
        "KNN",
        "Iterative"
    ],
    "Main Use": [
        "Few missing rows",
        "Very high missing column",
        "Symmetric numerical data",
        "Skewed numerical data",
        "Categorical data",
        "Meaningful default",
        "Time-series",
        "Ordered data",
        "Continuous ordered data",
        "Group differences",
        "Similar numerical records",
        "Feature relationships"
    ]
})

print(comparison)

        Technique                   Main Use
0       Drop Rows           Few missing rows
1    Drop Columns   Very high missing column
2            Mean   Symmetric numerical data
3          Median      Skewed numerical data
4            Mode           Categorical data
5        Constant         Meaningful default
6    Forward Fill                Time-series
7   Backward Fill               Ordered data
8   Interpolation    Continuous ordered data
9     Group-Based          Group differences
10            KNN  Similar numerical records
11      Iterative      Feature relationships


## Result / Observation

Different techniques solve different missing-data problems.

The method should be selected based on the characteristics of the dataset rather than applying one technique to every column.

# 21. Why We Should Not Always Use Mean Imputation

## Step 1 – Understand Concept

Mean imputation is simple, but it is not suitable for every numerical column.

It can cause problems when:

- Data is highly skewed
- Strong outliers exist
- Missingness is not random
- The feature has important relationships with other variables
- A large percentage of values are missing

Mean imputation can also reduce the natural variability of the data.


 ## Step 2 – Demonstrate Concept

Consider:

20, 25, 30, 35, 500

The value 500 is an extreme value.

The mean is heavily influenced by 500.

If we replace a missing value with this mean, the replacement may not represent a typical customer.

Median would usually be more robust in this situation.


## Step 3 – Implement the Concept

In [21]:
df = pd.DataFrame({
    "Income": [20000, 25000, 30000, 35000, 500000, np.nan]
})

mean_value = df["Income"].mean()
median_value = df["Income"].median()

print("Mean:", mean_value)
print("Median:", median_value)

df_mean = df.copy()
df_mean["Income"] = df_mean["Income"].fillna(mean_value)

df_median = df.copy()
df_median["Income"] = df_median["Income"].fillna(median_value)

print("\nMean Imputation:")
print(df_mean)

print("\nMedian Imputation:")
print(df_median)

Mean: 122000.0
Median: 30000.0

Mean Imputation:
     Income
0   20000.0
1   25000.0
2   30000.0
3   35000.0
4  500000.0
5  122000.0

Median Imputation:
     Income
0   20000.0
1   25000.0
2   30000.0
3   35000.0
4  500000.0
5   30000.0


## Result / Observation

The extreme value significantly increases the mean.

The median is less affected by the outlier.

Therefore, median imputation can be a better choice for skewed numerical data.

The key point is:

**Do not automatically use mean imputation for every missing numerical value.**

# 22. Missing Value Handling – Decision Process


## Step 1 – Understand Concept

A good missing-value workflow is:

1. Detect missing values.
2. Calculate missing percentage.
3. Understand why values are missing.
4. Check MCAR, MAR, or MNAR patterns where possible.
5. Check data type.
6. Check distribution and outliers.
7. Select an appropriate technique.
8. Apply the technique.
9. Validate the result.
10. Document the decision.


## Step 2 – Demonstrate Concept

Example:

MonthlyCharges has 3% missing values.

Analysis:

- Numerical feature
- Slightly skewed
- Contains some outliers

Decision:

Median imputation may be more appropriate than mean imputation.

Documentation:

Problem:
3% MonthlyCharges values are missing.

Analysis:
The feature is numerical and contains outliers.

Technique:
Median imputation.

Reason:
Median is less affected by extreme values.

Impact:
Missing values are handled while reducing the effect of outliers.



## Step 3 – Implement the Concept

In [22]:
df = pd.DataFrame({
    "MonthlyCharges": [50, 60, 65, np.nan, 70, 500]
})

print("Missing percentage:")
print(df["MonthlyCharges"].isna().mean() * 100)

median_value = df["MonthlyCharges"].median()

df["MonthlyCharges"] = df["MonthlyCharges"].fillna(median_value)

print("\nAfter Median Imputation:")
print(df)

Missing percentage:
16.666666666666664

After Median Imputation:
   MonthlyCharges
0            50.0
1            60.0
2            65.0
3            65.0
4            70.0
5           500.0


## Result / Observation

The missing value was handled using median imputation because the feature contains an extreme value.

The preprocessing decision was based on the characteristics of the data rather than automatically applying mean imputation.

# Final Summary – Missing Value Handling

In this notebook, I learned how to identify, analyze, and handle missing values.

## Missingness Concepts

- Missing Completely At Random (MCAR)
- Missing At Random (MAR)
- Missing Not At Random (MNAR)

## Detection

- `isna()`
- `isnull()`
- Missing value count
- Missing value percentage

## Basic Handling

- Dropping rows
- Dropping columns
- Mean imputation
- Median imputation
- Mode imputation
- Constant value imputation

## Advanced Handling

- Forward fill
- Backward fill
- Interpolation
- Group-based imputation
- KNN imputation
- Iterative imputation

## Key Learning

There is no single best technique for all missing values.

The technique should be selected based on:

- Data type
- Missing percentage
- Distribution
- Outliers
- Missingness pattern
- Feature relationships
- Business meaning

Mean imputation should not be applied blindly to every numerical feature.

Every preprocessing decision should be documented using:

1. Problem
2. Analysis
3. Technique Selected
4. Reason
5. Implementation
6. Result
7. Impact